
# ⚡ Human-Like Picker — **Top-20 Fast (Parallel + Persistent Cache)**

Optimized for your **32 GB RAM** machine:
- **Parallel engine workers** (each worker runs its own Stockfish with `Threads=1`)
- **Persistent gzip cache** for `fen → top20` so re-runs only compute misses
- **Single-shot MultiPV=20** per **unique FEN** (tiny movetime)
- Keep positions even if human move isn’t in top-20 → `in_top20` flag

You control speed via:
- `ENGINE_MOVETIME_MS` (15–25ms typical)
- `DEFAULT_WORKERS` (try 8–12 on a beefy CPU)
- `ENGINE_HASH_MB` (set per engine; with 8 workers × 256 MB ≈ 2 GB total engine hash)
- `keep_max_ply` (40 for speed; raise later)


In [1]:

# --- Config tuned for 32GB RAM ---
import os
ENGINE_MOVETIME_MS = 15     # 15–25ms; raise for a tad more coverage
ENGINE_THREADS     = 1      # 1 per engine; better throughput across workers
ENGINE_HASH_MB     = 256    # per engine; 8 workers -> ~2GB hash usage
DEFAULT_WORKERS    = 12      # try 8–12 depending on CPU
DEFAULT_NUM_BATCHES= 64     # chunking for smooth progress in parallel

ENGINE_PATH = os.getenv("ENGINE_PATH") or r"E:\Projects\AIP\stockfish\stockfish-windows-x86-64-avx2.exe"

# --- Hardcoded SQL settings (training only) ---
SQL_SERVER   = "tcp:64squares.database.windows.net,1433"  # tcp + port avoids Named Pipes
SQL_DB       = "64Squares"
SQL_USER     = "Squares"
SQL_PASSWORD = "Chess@123"
SQL_DRIVER   = "ODBC Driver 18 for SQL Server"

import pyodbc
cs = (
    f"DRIVER={{{SQL_DRIVER}}};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DB};"
    f"UID={SQL_USER};"
    f"PWD={SQL_PASSWORD};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)
# Smoke test: open + close
_conn = pyodbc.connect(cs)
print("✅ Connected to", SQL_SERVER, "/", SQL_DB)
_conn.close()



✅ Connected to tcp:64squares.database.windows.net,1433 / 64Squares


In [2]:

# --- timing + tqdm ---
%pip -q install tqdm

from tqdm.auto import tqdm
import time, math, threading, json, gzip
from contextlib import contextmanager
from collections import defaultdict

class TimeStats:
    def __init__(self):
        self.lock = threading.Lock()
        self.totals = defaultdict(float)
        self.counts = defaultdict(int)
        self.samples = defaultdict(list)

    @contextmanager
    def section(self, name, *, sample=False):
        t0 = time.perf_counter()
        try:
            yield
        finally:
            dt = time.perf_counter() - t0
            with self.lock:
                self.totals[name] += dt
                self.counts[name] += 1
                if sample: self.samples[name].append(dt)

    def summary(self):
        rows = []
        for k in sorted(self.totals):
            tot = self.totals[k]; n = max(1,self.counts[k])
            avg = tot/n
            p50=p90=p99=None
            if self.samples[k]:
                arr = sorted(self.samples[k])
                def pct(p): 
                    i = min(len(arr)-1, max(0, int(p*len(arr))-1))
                    return arr[i]
                p50, p90, p99 = pct(0.50), pct(0.90), pct(0.99)
            rows.append((k, self.counts[k], tot, avg, p50, p90, p99))
        return rows

TIMES = TimeStats()

def print_time_summary():
    print("\n=== Timing Summary ===")
    for k,n,tot,avg,p50,p90,p99 in TIMES.summary():
        line = f"{k:18s} n={n:<6d} total={tot:7.2f}s avg={avg*1000:6.1f}ms"
        if p50 is not None:
            line += f" p50={p50*1000:5.1f}ms p90={p90*1000:5.1f}ms p99={p99*1000:5.1f}ms"
        print(line)


Note: you may need to restart the kernel to use updated packages.


c:\Users\ppava\anaconda3\envs\chess-bot-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- Replace StockfishServiceSync with a sync wrapper using the 'stockfish' package ---
from pathlib import Path
from stockfish import Stockfish

class StockfishServiceSync:
    def __init__(self, path=None):
        from shutil import which
        self.path = path or ENGINE_PATH or which("stockfish")
        self.sf = None

    def open(self):
        p = Path(self.path)
        if not p.exists():
            raise FileNotFoundError(f"ENGINE_PATH invalid: {self.path!r}")
        self.sf = Stockfish(
            path=str(p),
            parameters={
                "Threads": ENGINE_THREADS,
                "Hash": ENGINE_HASH_MB,
                "MultiPV": 20,          # we’ll still request exactly what we want per call
            },
        )
        self.sf.set_depth(10)  # try 10 first; bump to 12 if coverage drops

    def get_top_moves(self, fen: str, n: int = 20):
        # Synchronous call; no asyncio; works fine in notebooks/threads on Windows
        self.sf.set_fen_position(fen)
        try:
            # Most recent 'stockfish' package exposes get_top_moves
            top = self.sf.get_top_moves(n)
        except AttributeError:
            # If your installed version lacks get_top_moves, upgrade:
            #   %pip install --upgrade stockfish
            raise RuntimeError("Your 'stockfish' package version lacks get_top_moves(n). Please upgrade it.")

        # Normalize fields to match our dataset schema
        norm = []
        for i, mv in enumerate(top or []):
            # mv example: {'Move': 'e2e4', 'Centipawn': 22, 'Mate': 0}
            norm.append({
                "uci": mv.get("Move"),
                "score_cp": mv.get("Centipawn"),
                "mate": mv.get("Mate") or 0,
                "depth": 0,        # the 'stockfish' wrapper doesn’t return depth/pv
                "pv": "",          # (optional) you can leave pv empty in this fast path
            })
        return {"top_moves": norm[:n]}

    def close(self):
        try:
            if self.sf:
                # The 'stockfish' wrapper cleans up its process when gc’d, but be explicit:
                self.sf = None
        except Exception:
            pass


In [4]:

# --- SQL + PGN parsing helpers ---
import pyodbc, io, re, pandas as pd, numpy as np, chess, chess.pgn

TC_RE  = re.compile(r"^\s*(\d+)(?:\+(\d+))?\s*$")
CLK_RE = re.compile(r"\[\s*%clk\s+([0-9:]+)\s*\]")

def clock_to_ms(clk: str):
    if not clk: return None
    parts = [int(p) for p in clk.split(":")]
    if   len(parts)==2: h,m,s = 0, parts[0], parts[1]
    elif len(parts)==3: h,m,s = parts
    else: return None
    return ((h*60+m)*60+s)*1000

def parse_timecontrol(tc: str):
    if not tc: return (None, None)
    m = TC_RE.match(tc.strip())
    if not m: return (None, None)
    return int(m.group(1))*1000, int(m.group(2) or 0)*1000

def pgntxt(start_fen, movetext):
    headers = ['[Event "-" ]','[Site "-" ]','[Date "????.??.??"]','[Round "-" ]',
               '[White "-" ]','[Black "-" ]','[Result "*" ]','[SetUp "1"]', f'[FEN "{start_fen}"]']
    body = movetext.strip()
    if not body.endswith(("1-0","0-1","1/2-1/2","*")): body += " *"
    return "\n".join(headers) + "\n\n" + body + "\n"

def iter_plies(start_fen, movetext_full, timecontrol):
    base_ms, inc_ms = parse_timecontrol(timecontrol or "")
    game = chess.pgn.read_game(io.StringIO(pgntxt(start_fen, movetext_full)))
    if not game: return
    board = game.board()
    prev_post = {True: None, False: None}
    ply = 0
    for node in game.mainline():
        if node.move is None: continue
        fen_before = board.fen()
        side = board.turn
        uci = node.move.uci()
        board.push(node.move)
        ply += 1
        post_ms = None
        if node.comment:
            m = CLK_RE.search(node.comment)
            if m: post_ms = clock_to_ms(m.group(1))
        think_ms = None
        if post_ms is not None and base_ms is not None:
            pre = base_ms if prev_post[side] is None else max(prev_post[side] + (inc_ms or 0), 0)
            d = pre - post_ms
            if 0 <= d <= 10*60*1000: think_ms = d
        prev_post[side] = post_ms
        yield {"ply": ply, "fen": fen_before, "side": "w" if side else "b",
               "human_uci": uci, "think_ms": think_ms}

def sql_fetch_games(elo_min, elo_max, limit_rows):
    cs = (
        f"DRIVER={{{SQL_DRIVER}}};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DB};"
        f"UID={SQL_USER};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
    )
    with TIMES.section("sql_connect"):
        conn = pyodbc.connect(cs)
        cur  = conn.cursor()
    with TIMES.section("sql_query"):
        cur.execute(f'''
            SELECT TOP {limit_rows}
              core.game_pk,
              COALESCE(NULLIF(core.start_fen,''),'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1') AS start_fen,
              COALESCE(core.timecontrol,'') AS timecontrol,
              txt.pgn_movetext_full
            FROM dbo.game_core core
            JOIN dbo.game_text txt ON txt.game_pk = core.game_pk
            WHERE txt.pgn_movetext_full IS NOT NULL
              AND core.elo_avg BETWEEN ? AND ?
            ORDER BY core.game_pk;
        ''', (elo_min, elo_max))
        rows = cur.fetchall()
    cur.close(); conn.close()
    return rows

def collect_unique_positions(rows, keep_max_ply, max_positions):
    uniq = set()
    pos = []
    for r in rows:
        for pl in iter_plies(r.start_fen, r.pgn_movetext_full, r.timecontrol):
            if pl["ply"] > keep_max_ply: break
            fen = pl["fen"]
            if fen not in uniq:
                uniq.add(fen)
                pos.append(pl)
                if len(pos) >= max_positions * 2:  # slack for filtering later
                    return pos
    return pos


In [5]:

# --- Persistent cache helpers ---
from pathlib import Path

CACHE_PATH = Path("data/fen_top20_cache.json.gz")

def load_fen_cache(path=CACHE_PATH):
    if path.exists():
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_fen_cache(fmap, path=CACHE_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(fmap, f)


In [6]:

# --- Parallel evaluation of unique FENs ---
from concurrent.futures import ThreadPoolExecutor, as_completed

def eval_unique_fens_parallel(pos_list, *, shortlist_n=20, workers=DEFAULT_WORKERS, num_batches=DEFAULT_NUM_BATCHES):
    # build worklist of unique FENs
    fens = []
    seen = set()
    for pl in pos_list:
        fen = pl["fen"]
        if fen not in seen:
            seen.add(fen)
            fens.append(fen)

    # load cache and filter
    fen_map = load_fen_cache()
    todo = [fen for fen in fens if fen not in fen_map]
    if not todo:
        return fen_map

    # chunk work
    num_batches = max(workers, num_batches)
    chunk_size  = max(1, math.ceil(len(todo)/num_batches))
    chunks      = [todo[i:i+chunk_size] for i in range(0, len(todo), chunk_size)]

    def _worker(chunk):
        svc = StockfishServiceSync(); svc.open()
        out = {}
        for fen in chunk:
            with TIMES.section("engine_eval", sample=True):
                res = svc.get_top_moves(fen, n=shortlist_n)
            out[fen] = res.get("top_moves", [])
        try: svc.close()
        except: pass
        return out

    with ThreadPoolExecutor(max_workers=workers) as ex, \
         tqdm(total=len(chunks), desc=f"Engine eval (workers={workers})", unit="chunk") as tq:
        futs = [ex.submit(_worker, ch) for ch in chunks]
        for fut in as_completed(futs):
            out = fut.result()
            fen_map.update(out)
            tq.update(1)

    # persist merged cache
    save_fen_cache(fen_map)
    return fen_map


In [7]:

# --- Build dataset (flag-if-missing) ---
def build_top20_fast_parallel(*,
    elo_min=2200, elo_max=2400,
    max_positions=2000, keep_max_ply=40,
    shortlist_n=20,
    workers=DEFAULT_WORKERS, num_batches=DEFAULT_NUM_BATCHES,
    out_parquet="data/candidates_top20_fast_parallel.parquet"
):
    import pandas as pd
    from pathlib import Path

    rows = sql_fetch_games(elo_min, elo_max, limit_rows=max_positions*3)
    pos_list = collect_unique_positions(rows, keep_max_ply, max_positions)
    print(f"[INFO] unique positions collected: {len(pos_list)}")

    fen_map = eval_unique_fens_parallel(pos_list, shortlist_n=shortlist_n, workers=workers, num_batches=num_batches)

    recs = []
    kept = 0
    for pl in tqdm(pos_list, desc="Record build", unit="pos"):
        fen, side, human_uci, think_ms, ply = pl["fen"], pl["side"], pl["human_uci"], pl["think_ms"], pl["ply"]
        eng = fen_map.get(fen, [])
        uci_list = [m["uci"] for m in eng]
        in_top20 = 1 if human_uci in uci_list else 0
        eng20 = eng[:shortlist_n]
        if len(eng20) == 0:
            continue
        group_id = f"{hash(fen)}:{kept}"
        best_cp = max([m.get("score_cp") for m in eng20 if m.get("score_cp") is not None], default=None)
        for m in eng20:
            uci = m["uci"]
            recs.append({
                "group_id": group_id,
                "fen": fen,
                "side": 1 if side=="w" else 0,
                "uci": uci,
                "label_choice": 1 if uci == human_uci else 0,
                "human_move_uci": human_uci,
                "human_think_ms": think_ms,
                "in_top20": in_top20,
                "ply": ply,
                "cp": m.get("score_cp") or 0,
                "mate": m.get("mate") or 0,
                "depth": m.get("depth") or 0,
                "pv_len": len(m.get("pv","").split()) if m.get("pv") else 0,
                "cp_to_best": (0 if (best_cp is None or m.get("score_cp") is None) else best_cp - m["score_cp"]),
            })
        kept += 1
        if kept >= max_positions:
            break

    df = pd.DataFrame.from_records(recs)
    Path(out_parquet).parent.mkdir(parents=True, exist_ok=True)
    with TIMES.section("parquet_write"):
        df.to_parquet(out_parquet, index=False)

    groups = df.groupby("group_id")["in_top20"].max().sum() if len(df) else 0
    print(f"[OK] wrote {len(df)} rows to {out_parquet} (groups={df['group_id'].nunique() if len(df) else 0})")
    print(f"[COVERAGE] human-in-top20 groups: {groups}/{df['group_id'].nunique() if len(df) else 0}")
    print_time_summary()
    return df


In [8]:

# === Example run ===
df_fast = build_top20_fast_parallel(
    elo_min=2200, elo_max=2400,
    max_positions=1000,
    keep_max_ply=40,
    shortlist_n=20,
    workers=14, num_batches=64,
    out_parquet="data/candidates_top20_fast_parallel.parquet"
)
df_fast.head(10)


[INFO] unique positions collected: 2000


Record build:  50%|████▉     | 999/2000 [00:00<00:00, 70945.46pos/s]


[OK] wrote 19706 rows to data/candidates_top20_fast_parallel.parquet (groups=1000)
[COVERAGE] human-in-top20 groups: 972/1000

=== Timing Summary ===
engine_eval        n=2000   total=6128.32s avg=3064.2ms p50=3087.8ms p90=5104.3ms p99=7999.3ms
parquet_write      n=1      total=   0.12s avg= 117.7ms
sql_connect        n=1      total=   0.01s avg=  13.4ms
sql_query          n=1      total=   1.78s avg=1783.8ms


,group_id,fen,side,uci,label_choice,human_move_uci,human_think_ms,in_top20,ply,cp,mate,depth,pv_len,cp_to_best
0,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,e2e4,0,d2d4,0,1,1,36,0,0,0,0
1,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,d2d4,1,d2d4,0,1,1,20,0,0,0,16
2,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,g1f3,0,d2d4,0,1,1,19,0,0,0,17
3,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,e2e3,0,d2d4,0,1,1,18,0,0,0,18
4,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,c2c4,0,d2d4,0,1,1,17,0,0,0,19
5,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,g2g3,0,d2d4,0,1,1,17,0,0,0,19
6,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,b1c3,0,d2d4,0,1,1,9,0,0,0,27
7,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,c2c3,0,d2d4,0,1,1,8,0,0,0,28
8,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,a2a3,0,d2d4,0,1,1,4,0,0,0,32
9,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,d2d3,0,d2d4,0,1,1,-8,0,0,0,44
